# Exercise set 9

The main goal of this exercise is to gain practical experience with using penalised B-splines for signal processing. This can be used to clean up noisy signals and compute their derivatives.

**Learning Objectives:**

After completing this exercise set, you will be able to:

- Apply penalised B-splines to filter noise from signals.
- Perform numerical differentiation and integration of noisy signals using splines.

**To get the exercise approved, complete the following problems:**

- [9.1(a)](#9.1(a)): To show that you can smooth a noisy signal using splines.
- [9.1(c)](#9.1(c)) and [9.1(e)](#9.1(e)): To show that you can perform numerical differentiation and integration using splines.

**Files required for this exercise:**
* [Exercise 9.1](#Exercise-9.1): [signal.txt](signal.txt) and [signal_noise.txt](signal_noise.txt)


Please ensure that these files are saved in the same directory as this notebook.

## Exercise 9.1

In this exercise, we will smooth, differentiate and also integrate a noisy signal. We will use a test signal generated from the following analytical function:

$$y(t) = \sin (8t) - 1.8t^2 + 0.5t^3.$$

The noise-free signal data is available in the file [signal.txt](signal.txt). The file contains two columns: the first column represents time ($t$), and the second column represents the signal $y(t)$.

A noisy version of this signal is provided in the file [signal_noise.txt](signal_noise.txt), which also contains two columns: time ($t$) and the signal $y(t)$ with added noise.

The example code below demonstrates how to load and plot the raw data:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="ticks", context="notebook", palette="colorblind")

t_clean, y_clean = np.loadtxt("signal.txt", unpack=True)
t_noisy, y_noisy = np.loadtxt("signal_noise.txt", unpack=True)

fig, ax = plt.subplots(constrained_layout=True)
ax.plot(t_noisy, y_noisy, label="With noise (signal_noise.txt)", alpha=0.5)
ax.plot(t_clean, y_clean, label="Without noise (signal.txt)", lw=3)
ax.set(xlabel="Time (t)", ylabel="Signal (y(t))")
ax.legend()
sns.despine(fig=fig)

### 9.1(a)

**Task: Smooth the signal with noise using a penalised quadratic B-spline. Compare the result to the noise-free signal by plotting both signals.**

**Hint:** You can create the B-spline basis set using the `bbase` function provided in the code cell below. You can use this function to create the smoothed signal by completing the following steps:

1. First calculate the B-spline design matrix `B2`:
   ```python
   ndx = 20  # Number of B-spline segments
   degree = 2  # B-spline degree
   B2, knots2, centres2 = bbase(t_noisy, ndx=ndx, deg=degree)
   ```

2. Create the penalty matrix `Dn` (typically, we use D2 or D3 for better interpolation):
   ```python
   order = 2  # Order of the penalty (1, 2, or 3)
   Dn = getD(order, B2.shape[1])
   ```

3. Create the augmented matrix `A2` that combines the design matrix `B2` obtained from `bbase` with the scaled penalty matrix `sqrt(lamb)*Dn`, and create the augmented column vector `yaug` that combines the y-values from the data set with a vector of zeros equal in length to the number of rows of `Dn`:
   ```python
   # Create the augmented matrix A2
   lamb = 1  # Adjust this to control smoothing
   A2 = np.vstack([B2, np.sqrt(lamb) * Dn])
   # Create the augmented column vector yaug
   yaug = np.concatenate([y_noisy, np.zeros(Dn.shape[0])])
   ```

4. Find the coefficients that best fit the data set by solving the linear equation `yaug = A2*beta2`:
   ```python
   beta2 = np.linalg.lstsq(A2, yaug, rcond=None)[0]
   ```

5. Find the smoothed data `y_smooth` by calculating `y_smooth = B2*beta2`:
   ```python
   # Calculate the smoothed data:
   y_smooth = B2 @ beta2
   ```

In [ ]:
from scipy.sparse import diags


def tpower(x, t, n):
    """Generate degree-n truncated power function."""
    y = (x - t) ** n * (x > t)
    return y


def getD(order, nB):

    # Generate difference stencil (Pascal alternating)
    seed = np.hstack([np.zeros(order), 1, np.zeros(order)])

    # create B-spline stencil
    stencil = (-1) ** order * np.diff(seed, n=order)

    # replicate stencil for each row
    stencil = np.tile(stencil, (nB - order, 1))

    # Create banded difference matrix
    offsets = np.arange(order + 1)
    D = diags(stencil.T, offsets, shape=(nB - order, nB)).todense()  # ensure dense
    D = np.array(D)  # convert to ndarray explicitly

    return D


def bbase(x, xl=None, xr=None, ndx=20, deg=3):
    """Generate degree-p truncated power function."""
    if xl is None:
        xl = np.min(x)
    if xr is None:
        xr = np.max(x)
    # Generate knot sequence (extends outside boundaries)
    dx = (xr - xl) / ndx
    knots = np.arange(xl - deg * dx, xr + deg * dx + dx, dx)

    # Calculate centre positions for the BBFs
    halfwidth = (deg + 1) * dx / 2
    nBBF = deg + ndx
    centres = knots[:nBBF] + halfwidth

    # Compute expanded TPF design matrix
    T = np.zeros((len(x), len(knots)))
    for i in range(len(knots)):
        T[:, i] = tpower(x, knots[i], deg)

    # Generate difference matrix
    D = getD(deg + 1, ndx + 2 * deg + 1)

    # Compute (unnormalised) BBF design matrix, correcting for sign of D
    B = (-1) ** (deg + 1) * T @ D.T

    # Normalise BBF design matrix
    B = B / np.sum(B, axis=1, keepdims=True)
    return B, knots, centres

In [ ]:
ndx = 20  # Number of B-spline segments
degree = 2  # B-spline degree
B2, knots2, centres2 = bbase(t_noisy, ndx=ndx, deg=degree)

order = 2  # Order of the penalty (1, 2, or 3)
Dn = getD(order, B2.shape[1])

lamb = 0.3  # Adjust this to control smoothing
A2 = np.vstack([B2, np.sqrt(lamb) * Dn])
# Create the augmented column vector yaug
yaug = np.concatenate([y_noisy, np.zeros(Dn.shape[0])])
beta2 = np.linalg.lstsq(A2, yaug, rcond=None)[0]
y_smooth = B2 @ beta2

In [ ]:
from sklearn.metrics import root_mean_squared_error

In [ ]:
fig, axes = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
# Plot signals
axes[0].plot(t_noisy, y_noisy, label="Signal with noise", alpha=0.2, color="0.2")
axes[0].plot(t_clean, y_clean, label="Signal without noise", color="0.5", ls="-")
axes[0].plot(t_noisy, y_smooth, label="Smoothed Signal", ls=":")
axes[0].set(xlabel="t", ylabel="y(t)")
axes[0].legend()

# Plot the  noise-free vs smoothed
axes[1].plot(y_clean, y_smooth, marker="o", markerfacecolor="none", ls="none")
rmse = root_mean_squared_error(y_clean, y_smooth)
axes[1].set(xlabel="Signal without noise", ylabel="Smoothed Signal")
axes[1].set_title(f"RMSE = {rmse:.3f}", loc="left")

# Add the x=y line:
axes[1].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)
sns.despine(fig=fig)

#### Your answer to question 9.1(a): How does the smoothed signal compare to the noise-free signal?
The smoothed signal compares well to the noise-free signal, and the smoothing process has removed much of the noise. The smoothed signal follows the trend of the original signal, and the root mean squared error (RMSE) between the smoothed and noise-free signal is 0.232 (which is a good fit since the noise-free signal has values between -11 and 1). There are some small deviations (e.g., one "hump" is shifted around -1), but the overall fit is good.

### 9.1(b)

**Task: We could attempt to estimate the derivative of the noisy signal directly using a finite difference approximation, such as the forward difference method:**

$$y'(t_i) \approx \frac{y(t_{i+1}) - y(t_i)}{t_{i+1} - t_i}$$

**Explain the potential problem(s) associated with this approach when applied to a signal containing substantial noise.**

#### Your answer to question 9.1(b): What are the potential problem(s) when using the finite difference approximation directly with a noisy signal?

The noise makes the signal jump up and down a lot between points. If we take the difference between these "jumping" points, we can get large fluctuations. Assuming that the size of the error is $\varepsilon_i$, so that the measured points are $y(t_i) =y^\ast (t_i) \pm \varepsilon_i$, where $y^\ast (t_i)$ is the true noise-free value, then we could potentially compute the derivative as (worst-case),

$$y'(t_i) \approx \frac{(y^\ast(t_{i+1}) - y^\ast(t_i)) +  (\varepsilon_{i+1}+\varepsilon_i)}{t_{i+1} - t_i},$$

meaning that we get an error of approximately

$$\frac{\varepsilon_{i+1}+\varepsilon_i}{t_{i+1} - t_i}.$$

For small time differences and large errors (compared to $y^\ast$), this can significantly increase and dominate the derivative. The noise gets amplified, and the real derivative gets lost in the noise.

### 9.1(c)

**Task: Calculate the derivative of the smoothed signal from [9.1(a)](#9.1(a)) using the B-spline representation. Compare the derivative to the analytical derivative of the noise-free signal (by plotting both derivatives).**

**Hint:**

1. The analytical derivative of the noise-free signal is:

   $$ y^\prime(t) = 8\cos(8t) - 3.6t + 1.5t^2,$$

   with Python: 
   
   ```python
   def analytical_derivative(t):
       """Calculate the analytical derivative of the noise-free signal."""
       return 8 * np.cos(8 * t) - 3.6 * t + 1.5 * t**2
   ```

2. You can find the first derivative of the B-spline using:
   ```python
   (1 / h) * B1 @ (D1 @ beta2)
   ```
   where:
   
   * `h` is the spacing (same as `dx` in `bbase`):
      ```python
      h = (t_noisy.max() - t_noisy.min()) / ndx
      ```
   * `B1` is the linear B-spline design matrix (one degree less than the quadratic you used for finding the coefficients `beta2`):
      ```python
      B1, knots1, centres1 = bbase(t_noisy, ndx=ndx, deg=1)
      ```
   
   * `D1` is the first order derivative matrix given by
      ```python
      D1 = getD(order=1, nB=B2.shape[1])
      ```

    A combined example is:
   ```python
   h = (t_noisy.max() - t_noisy.min()) / ndx
   D1 = getD(order=1, nB=B2.shape[1])
   B1, knots1, centres1 = bbase(t_noisy, ndx=ndx, deg=1)
   derivative_smooth = (1 / h) * B1 @ (D1 @ beta2)
   ```

In [ ]:
def analytical_derivative(t):
    """Calculate the analytical derivative of the noise-free signal."""
    return 8 * np.cos(8 * t) - 3.6 * t + 1.5 * t**2


derivative_noisefree = analytical_derivative(t_noisy)

In [ ]:
h = (t_noisy.max() - t_noisy.min()) / ndx
D1 = getD(order=1, nB=B2.shape[1])
B1, knots1, centres1 = bbase(t_noisy, ndx=ndx, deg=1)
derivative_smooth = (1 / h) * B1 @ (D1 @ beta2)

In [ ]:
fig, axes = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))

axes[0].plot(t_noisy, derivative_noisefree, label="Analytical")
axes[0].plot(
    t_noisy,
    derivative_smooth,
    label="Numerical (spline)",
    ls="-.",
)
axes[0].set(xlabel="t", ylabel="y'(t)")
axes[0].legend()

# Plot the  noise-free vs smoothed
axes[1].plot(
    derivative_noisefree,
    derivative_smooth,
    marker="o",
    markerfacecolor="none",
    ls="none",
)
rmse = root_mean_squared_error(derivative_noisefree, derivative_smooth)
axes[1].set_title(f"RMSE = {rmse:.3f}", loc="left")
axes[1].set(
    xlabel="y'(t) (Analytical)",
    ylabel="y'(t) (Numerical (spline))",
)


# Add the x=y line:
axes[1].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)


sns.despine(fig=fig)

#### Your answer to question 9.1(c): How does the computed derivative compare to the analytical derivative?

The computed derivative shows a qualitative correspondence with the analytical derivative, accurately reflecting its overall trend, including the location of its local extrema.

Quantitatively, the Root Mean Squared Error (RMSE) provides a measure of the average deviation, calculated to be  1.728. Considering the RMSE value in the context of the analytical derivative's range (-11 to 17), this value suggests a reasonably close agreement between the two derivatives.

### 9.1(d)

**Task: Smooth the signal and obtain its derivative using a Savitzky-Golay filter. Compare the smoothness and accuracy of both methods (Savitzky-Golay and B-splines) by plotting the results.**

**Hint:** The Savitzky-Golay filter can be applied as follows:
```python
from scipy.signal import savgol_filter

y_smooth_sg = savgol_filter(
    y_noisy,
    window_length=51,  # Length of window to use for smoothing
    polyorder=3,  #  Polynomial order to use.
)

delta_t = t_noisy[1] - t_noisy[0]
derivative_smooth_sg = savgol_filter(
    y_noisy,
    delta=delta_t,  # Sample spacing, needed for the derivative.
    window_length=101,  # Length of window to use for smoothing
    polyorder=3,  #  Polynomial order to use.
    deriv=1,  # Compute the first (1) derivative.
)
# Note: Please experiment with different window lengths and orders for the polynomial.
```

In [ ]:
from scipy.signal import savgol_filter

y_smooth_sg = savgol_filter(
    y_noisy,
    window_length=101,  # Length of window to use for smoothing
    polyorder=5,  #  Polynomial order to use.
)

delta_t = t_noisy[1] - t_noisy[0]
derivative_smooth_sg = savgol_filter(
    y_noisy,
    delta=delta_t,  # Sample spacing, needed for the derivative.
    window_length=101,  # Length of window to use for smoothing
    polyorder=5,  #  Polynomial order to use.
    deriv=1,  # Compute the first (1) derivative.
)

In [ ]:
fig, axes = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))

# Plot the  noise-free vs smoothed
axes[0].plot(t_noisy, y_noisy, label="Signal with noise", alpha=0.2, color="0.2")
axes[0].plot(t_clean, y_clean, label="Signal without noise", color="0.5", ls="-")
axes[0].plot(t_noisy, y_smooth, label="Smoothed Signal (spline)", ls=":")

axes[0].plot(
    t_noisy,
    y_smooth_sg,
    label="Smoothed Signal (Savitzky-Golay)",
    ls="--",
)
axes[0].set(xlabel="t", ylabel="y(t)")
axes[0].legend(fontsize="x-small")


rmse_1 = root_mean_squared_error(y_clean, y_smooth)
rmse_2 = root_mean_squared_error(y_clean, y_smooth_sg)

axes[1].plot(
    y_clean,
    y_smooth,
    label=f"Spline (RMSE = {rmse_1:.3f})",
    marker="o",
    markerfacecolor="none",
    ls="none",
)
axes[1].plot(
    y_clean,
    y_smooth_sg,
    label=f"Savitzky-Golay (RMSE = {rmse_2:.3f})",
    marker="s",
    markerfacecolor="none",
    ls="none",
)
axes[1].set(xlabel="Signal without noise", ylabel="Smoothed Signal")
axes[1].legend(loc="upper left", fontsize="x-small")

# Add the x=y line:
axes[1].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)
sns.despine(fig=fig)

In [ ]:
fig, axes = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
# Plot the  noise-free vs smoothed
axes[0].plot(t_noisy, derivative_noisefree, label="Analytical", color="0.5")
axes[0].plot(
    t_noisy,
    derivative_smooth,
    label="Numerical (spline)",
    ls="--",
)
axes[0].plot(
    t_noisy,
    derivative_smooth_sg,
    label="Numerical (Savitzky-Golay)",
    ls=":",
)
axes[0].set(xlabel="t", ylabel="y'(t)")
axes[0].legend(fontsize="x-small")


rmse_1 = root_mean_squared_error(derivative_noisefree, derivative_smooth)
rmse_2 = root_mean_squared_error(derivative_noisefree, derivative_smooth_sg)
axes[1].plot(
    derivative_noisefree,
    derivative_smooth,
    label=f"Spline (RMSE = {rmse_1:.3f})",
    marker="s",
    markerfacecolor="none",
    ls="none",
)
axes[1].plot(
    derivative_noisefree,
    derivative_smooth_sg,
    label=f"Savitzky-Golay (RMSE = {rmse_2:.3f})",
    marker="s",
    markerfacecolor="none",
    ls="none",
)
axes[1].legend(fontsize="x-small")
axes[1].set(
    xlabel="y'(t) (Analytical)",
    ylabel="y'(t) (Numerical)",
)
# Add the x=y line:
axes[1].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)
sns.despine(fig=fig)

#### Your answer to question 9.1(d): How do the results from applying the Savitzky-Golay filter compare to the B-splines results

Overall, the Savitzky-Golay filter and B-spline methods produce qualitatively similar results. Both methods effectively smooth the noisy signal and yield derivatives that closely resemble the analytical derivative.
Quantitatively, the root mean squared errors (RMSEs) between the smoothed and noise-free signals are also comparable, indicating similar levels of accuracy.

Both methods require parameter tuning (the `window_length` and `polyorder` for Savitzky-Golay and the smoothing factor for B-splines).

### 9.1(e)

**Task: Calculate the integral of the smoothed signal from [9.1(a)](#9.1(a)) using the B-spline representation. Compare the spline integral to the analytical integral of the noise-free signal (by plotting both integrals).**

**Hint:**
1. The analytical integral of the noise-free signal is:
   $$\int y(t)\, \mathrm{d}t = \int \left(\sin(8t) - 1.8t^2 + 0.5t^3\right)\, \mathrm{d}t $$
   $$ = -\frac{1}{8}\cos(8t) - 0.6t^3 + 0.125t^4 + C $$

   with Python (assuming $C = 0$): 
   ```python
   def analytical_integral(t):
       """Calculate the analytical integral of the noise-free signal."""
       return -(1 / 8) * np.cos(8 * t) - 0.6 * t**3 + 0.125 * t**4
   ```

2. You can find the coefficients for the (cubic B-spline) integral of the quadratic B-spline by solving the least squares problem $(D/h)\,\beta_{3} = \beta_{2}$, using
   ```python
   beta3 = np.linalg.lstsq((D / h), beta2, rcond=None)[0]
   ```
   where:

   * `D` is the first order derivative matrix for the cubic B-spline given by
      ```python
      D = getD(order=1, nB=B3.shape[1])
      ```

   * `B3` is the cubic B-spline design matrix (one degree more than the quadratic B-spline you used for finding the coefficients `B2`):
      ```python
      B3, knots3, centres3 = bbase(t_noisy, ndx=ndx, deg=3)
      ``` 

3. To get the spline integral ($\hat{y}$), you can then calculate $\hat{y} = (B3) \beta_{3}$:
   ```python
   integral = B3 @ beta3
   ```
   
4. The result from 3. will have an arbitrary constant of integration. We often want to set the integral at the left boundary to zero. The value of the spline at the left boundary is given by: 
   ```python
   integral_left = B3[0, :] @ beta3
   ```
   The corrected coefficients become:
   ```python
   beta3_corr = beta3 - integral_left
   ```
   The spline integral
   ```python
   integral_corr = B3 @ beta3_corr
   ```
   will now give a result of zero at the left boundary.

In [ ]:
def analytical_integral(t):
    """Calculate the analytical integral of the noise-free signal."""
    return -(1 / 8) * np.cos(8 * t) - 0.6 * t**3 + 0.125 * t**4


analytical_int = analytical_integral(t_noisy)
# Offset the integral so that it is zero at the left boundary:
analytical_int = analytical_int - analytical_integral(t_noisy[0])

In [ ]:
# For comparison, compute the integral using the trapezoidal rule:
from scipy.integrate import cumulative_trapezoid

trapz = cumulative_trapezoid(y_noisy, t_noisy, initial=0)

In [ ]:
B3, knots3, centres3 = bbase(t_noisy, ndx=ndx, deg=3)
D = getD(order=1, nB=B3.shape[1])
beta3 = np.linalg.lstsq((D / h), beta2, rcond=None)[0]

integral_left = B3[0, :] @ beta3
beta3_corr = beta3 - integral_left
integral_corr = B3 @ beta3_corr
integral = B3 @ beta3

In [ ]:
fig, axes = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))

axes[0].plot(t_noisy, analytical_int, label="Analytical")
axes[0].plot(
    t_noisy,
    integral_corr,
    label="Numerical (spline)",
    ls="-.",
)
axes[0].plot(
    t_noisy,
    trapz,
    label="Numerical (Trapezoidal rule)",
    ls=":",
)
axes[0].set(xlabel="t", ylabel="Integral")
axes[0].legend(fontsize="x-small")

# Plot the  noise-free vs smoothed

rmse1 = root_mean_squared_error(analytical_int, integral_corr)
rmse2 = root_mean_squared_error(analytical_int, trapz)

axes[1].plot(
    analytical_int,
    integral_corr,
    marker="o",
    markerfacecolor="none",
    ls="none",
    label=f"Spline (RMSE = {rmse1:.3f})",
)

axes[1].plot(
    analytical_int,
    trapz,
    marker="s",
    markerfacecolor="none",
    ls="none",
    label=f"Trapezoidal rule (RMSE = {rmse2:.3f})",
)

axes[1].legend(fontsize="x-small")
axes[1].set(
    xlabel="Analytical integral",
    ylabel="Numerical (spline) integral",
)

# Add the x=y line:
axes[1].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)


sns.despine(fig=fig)

#### Your answer to question 9.1(e): How does the computed integral compare to the analytical integral?

The computed spline integral is in close agreement with the analytical integral, and the Root Mean Squared Error (RMSE) is small (0.099). The integral computed with the trapezoidal rule is also in close agreement with the analytical integral (RMSE equal to 0.1). This reflects that integration (unlike differentiation) acts as a smoothing operation.